# 🛡️ UNSW-NB15 Cyber Threat Detection

Machine learning analysis of the UNSW-NB15 network intrusion dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Load UNSW-NB15 Training Data

In [ ]:
data = pd.read_csv('../data/UNSW_NB15_training-set.csv')

print('Dataset shape:', data.shape)
data.head()

## 2. Dataset Information

In [ ]:
print('Columns:')
print(data.columns.tolist())

print('\nMissing values:')
print(data.isnull().sum())

## 3. Attack Category Distribution

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=data, x='attack_cat', order=data['attack_cat'].value_counts().index)
plt.title('UNSW-NB15 Attack Category Distribution')
plt.xlabel('Attack Category')
plt.ylabel('Number of Records')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Prepare the Dataset

In [ ]:
data = data.dropna(subset=['attack_cat']).copy()
data['attack_cat'] = data['attack_cat'].astype(str).str.strip()

if 'id' in data.columns:
    data = data.drop(columns=['id'])

X = data.drop(columns=['attack_cat', 'label'], errors='ignore')
y = data['attack_cat']

## 5. Create a Manageable Stratified Sample

In [ ]:
if len(X) > 30000:
    X, _, y, _ = train_test_split(
        X,
        y,
        train_size=30000,
        random_state=42,
        stratify=y
    )

print('Working dataset size:', X.shape)

## 6. Feature Engineering

In [ ]:
categorical_features = X.select_dtypes(include=['object']).columns.tolist()
numeric_features = X.select_dtypes(exclude=['object']).columns.tolist()

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

## 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 8. Train Random Forest Classifier

In [ ]:
model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced',
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

print('✅ Model training completed.')

## 9. Evaluate the Model

In [ ]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print(f'Accuracy: {accuracy:.2%}')
print('\nClassification Report:')
print(classification_report(y_test, predictions, zero_division=0))

## 10. Confusion Matrix

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, predictions, labels=labels)

plt.figure(figsize=(10, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    xticklabels=labels,
    yticklabels=labels
)
plt.title('UNSW-NB15 Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## ✅ Conclusion

The Random Forest model was trained to classify network traffic into the attack categories available in the UNSW-NB15 training data.